![IITIS](pictures/logoIITISduze.png)

In [1]:
import numpy as np
from typing import NamedTuple
from math import inf
from tqdm import tqdm


class State(NamedTuple):
    configuration: list
    energy: float


# Funkcja pomocniczna, warto zauważyć że są prowadzone pewne modyfikacje
def calculate_energy(J: np.ndarray, h: np.ndarray, state: list):
    n = len(h)
    k = len(state)
    state = state + [0] * (n - k)
    state = np.array(state)
    return state @ J @ state.T + state @ h 


def branch_and_bound(J, h, num_states):
    
    n = len(h)
    states = [State([], inf)]
    
    for _ in tqdm(range(n), desc="Wykonywanie branch and bound"):
        # Branching
        temp = []
        for state in states:
            plus = state.configuration + [1]
            plus_energy = calculate_energy(J, h, plus)
            temp.append(State(plus, plus_energy))

            minus = state.configuration + [-1]
            minus_energy = calculate_energy(J, h, minus)
            temp.append(State(minus, minus_energy))
        
        # Bounding
        temp.sort(key=lambda x: x.energy)
        if len(temp) > num_states:
            temp = temp[:num_states]
        states = temp

    return states[0]

        

In [2]:
# wygenerujmy losową gęstą instancję
n = 20

scaling_func = np.vectorize(lambda x: 2*x-1)  # przesunięcie wartości z (0, 1) do (-1, 1)
J = np.triu(scaling_func(np.random.rand(n, n)))  # losowa gęsta macierz górnotrójkątna
np.fill_diagonal(J, 0.0)
h = scaling_func(np.random.rand(n))  # losowy wektor

from dimod import BinaryQuadraticModel
from dwave.samplers import SimulatedAnnealingSampler

bqm_instance = BinaryQuadraticModel(h, J, vartype="SPIN")
sampler= SimulatedAnnealingSampler()
sampleset = sampler.sample(bqm_instance, num_reads=1)
print(sampleset)


branch_and_bound(J, h, 10)

   0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 ... 19     energy num_oc.
0 +1 +1 -1 +1 -1 +1 -1 -1 -1 +1 -1 -1 -1 +1 -1 +1 -1 ... -1 -32.514529       1
['SPIN', 1 rows, 1 samples, 20 variables]


Wykonywanie branch and bound: 100%|██████████| 20/20 [00:00<00:00, 7500.54it/s]


State(configuration=[1, -1, -1, 1, -1, 1, 1, -1, -1, -1, -1, -1, 1, -1, -1, -1, -1, -1, 1, -1], energy=np.float64(-30.585858329529717))

In [5]:
import os
import pandas as pd

instance_path = os.path.join("instancje", "Pegasus", "P4_CBFM-P.txt")
# W pliku jest poza samą instancję jeszcze najlepsza znaleziona energia E = -469


def read_instance(path: os.PathLike):
    df = pd.read_csv(path, sep=" ", header=None, comment="#", names=["i", "j", "value"])

    n = max(df[["i", "j"]].max())
    h = np.zeros(n)
    J = np.zeros((n, n))
    
    for row in df.itertuples():
        if row.i == row.j:
            h[row.i - 1] = row.value
        elif row.i > row.j:
            J[row.j - 1, row.i - 1] = row.value  # by zachować górnotrójkątność
        else:
            J[row.i - 1, row.j - 1] = row.value
            
    return J, h

J, h = read_instance(instance_path)



solution = branch_and_bound(J, h, 10000)
print(solution.energy)

Wykonywanie branch and bound: 100%|██████████| 216/216 [01:46<00:00,  2.03it/s]

-465.0
